# Tutorial: MiniAutoGen Mini App Example

## Público
- Time de engenharia que quer ver a nova arquitetura em um fluxo de aplicação pequeno, executável e próximo de um cenário real.

## Pré-requisitos
- Binário `gemini` instalado e autenticado.
- Dependências do projeto disponíveis no ambiente do notebook.
- Familiaridade básica com `async` / `await`, stores e pipelines.

## Objetivos
- Montar um mini app real usando Gemini CLI como motor LLM.
- Persistir mensagens, runs e checkpoints.
- Inspecionar eventos e resultados do runtime novo.


## Roteiro

1. Setup e pré-checagem do Gemini CLI
2. Configuração do mini app com AgentAPIDriver
3. Pipeline de atendimento com driver backend
4. Montagem explícita da aplicação
5. Run único
6. Batch de runs
7. Inspeção de mensagens e eventos

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    return start

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

In [ ]:
def assert_gemini_cli_ready() -> dict[str, object]:
    binary = subprocess.run(
        ['bash', '-lc', 'command -v gemini'],
        capture_output=True,
        text=True,
        check=False,
    )
    gemini_path = binary.stdout.strip()
    if not gemini_path:
        raise RuntimeError('Binary `gemini` não encontrado no PATH.')

    last_error = None
    for attempt in range(1, 4):
        try:
            smoke = subprocess.run(
                ['gemini', '-m', 'gemini-2.5-pro', '--output-format', 'json'],
                input='Respond with exactly: NOTEBOOK-GEMINI-OK',
                capture_output=True,
                text=True,
                check=False,
                timeout=180,
            )
            if smoke.returncode != 0:
                last_error = f'Gemini CLI falhou no smoke test: {smoke.stderr}'
                continue
            if 'NOTEBOOK-GEMINI-OK' not in smoke.stdout:
                last_error = (
                    'Smoke test do Gemini CLI não retornou o texto esperado. '
                    f'stdout={smoke.stdout!r}'
                )
                continue
            return {
                'gemini_path': gemini_path,
                'smoke_test': 'NOTEBOOK-GEMINI-OK',
                'attempts_used': attempt,
                'stderr_excerpt': smoke.stderr.strip()[:200],
            }
        except subprocess.TimeoutExpired as exc:
            last_error = f'Timeout no smoke test do Gemini CLI após {exc.timeout} segundos.'

    raise RuntimeError(last_error or 'Falha desconhecida no smoke test do Gemini CLI.')

assert_gemini_cli_ready()


## 1. Configuração do mini app

Este exemplo usa a nova arquitetura de backend drivers: `AgentAPIClient` + `AgentAPIDriver` conectados ao gateway Gemini CLI via transporte ASGI embutido. O `PipelineRunner` e os stores separados continuam iguais.

In [ ]:
import httpx

from gemini_cli_gateway.app import app as gateway_app
from miniautogen.app.settings import MiniAutoGenSettings
from miniautogen.backends.agentapi import AgentAPIClient, AgentAPIDriver
from miniautogen.backends.models import SendTurnRequest, StartSessionRequest
from miniautogen.core.contracts import Message, RunContext
from miniautogen.core.events import InMemoryEventSink
from miniautogen.core.runtime import PipelineRunner
from miniautogen.policies import ExecutionPolicy
from miniautogen.stores import InMemoryCheckpointStore, InMemoryRunStore, SQLAlchemyMessageStore

DB_PATH = Path(tempfile.gettempdir()) / 'miniautogen-mini-app-example.db'
if DB_PATH.exists():
    DB_PATH.unlink()

os.environ['DATABASE_URL'] = f'sqlite+aiosqlite:///{DB_PATH}'
os.environ['MINIAUTOGEN_DEFAULT_PROVIDER'] = 'gemini-cli-gateway'
os.environ['MINIAUTOGEN_DEFAULT_MODEL'] = 'gemini-2.5-pro'
os.environ['MINIAUTOGEN_DEFAULT_TIMEOUT_SECONDS'] = '90'
os.environ['MINIAUTOGEN_GATEWAY_BASE_URL'] = 'http://gemini-gateway'

settings = MiniAutoGenSettings()

# Setup AgentAPIClient com transporte ASGI embutido (sem rede real)
client = AgentAPIClient(
    base_url='http://gemini-gateway',
    transport=httpx.ASGITransport(app=gateway_app),
    health_endpoint=None,
    max_retry_attempts=3,
    timeout_seconds=60.0,
)

# Setup AgentAPIDriver
driver = AgentAPIDriver(client=client, model='gemini-2.5-pro')

# Iniciar sessão com o backend
session = await driver.start_session(StartSessionRequest(backend_id='gemini'))

# Helper para extrair texto dos eventos do driver
async def ask_model(messages: list[dict]) -> str:
    async for event in driver.send_turn(
        SendTurnRequest(session_id=session.session_id, messages=messages)
    ):
        if event.type == 'message_completed':
            return event.payload.get('text', '')
    return ''

{
    'session_id': session.session_id,
    'capabilities': session.capabilities.enabled(),
    'model': settings.default_model,
    'timeout_seconds': settings.default_timeout_seconds,
}

## 2. Pipeline de atendimento

A lógica de negócio continua simples: classificar rota, persistir mensagens e pedir uma resposta ao Gemini CLI. A diferença é que agora usamos o `AgentAPIDriver` via `ask_model()` em vez de chamar `provider.generate_response()` diretamente.

In [ ]:
class SupportMiniAppPipeline:
    def __init__(self, driver, ask_model, message_store) -> None:
        self.driver = driver
        self.ask_model = ask_model
        self.message_store = message_store

    def classify_route(self, prompt: str) -> tuple[str, str]:
        lowered = prompt.lower()
        if 'fatura' in lowered or 'pagamento' in lowered:
            return 'billing', 'high'
        if 'erro' in lowered or 'falha' in lowered:
            return 'support', 'high'
        return 'general', 'normal'

    async def run(self, context: RunContext) -> dict[str, object]:
        payload = dict(context.input_payload or {})
        prompt = payload['prompt']
        customer_tier = payload.get('customer_tier', 'standard')
        route, priority = self.classify_route(prompt)

        await self.message_store.add_message(
            Message(
                sender_id='user',
                content=prompt,
                additional_info={
                    'run_id': context.run_id,
                    'route': route,
                    'customer_tier': customer_tier,
                },
            )
        )

        llm_prompt = (
            'Você é um assistente corporativo. '
            'Responda em português do Brasil, em no máximo duas frases, '
            f'classificando a solicitação como {route}.\n\nSolicitação: {prompt}'
        )
        reply = await self.ask_model(
            [{'role': 'user', 'content': llm_prompt}],
        )

        await self.message_store.add_message(
            Message(
                sender_id='support-agent',
                content=reply,
                additional_info={
                    'run_id': context.run_id,
                    'route': route,
                    'priority': priority,
                },
            )
        )

        context.execution_state['route'] = route
        context.execution_state['priority'] = priority

        return {
            'run_id': context.run_id,
            'route': route,
            'priority': priority,
            'customer_tier': customer_tier,
            'reply': reply,
        }

## 3. Montagem explícita

Aqui aparece a composição real do mini app: mensagens em SQLAlchemy local, run/checkpoint em memória e `AgentAPIDriver` conectado ao gateway Gemini CLI via ASGI.

In [ ]:
message_store = SQLAlchemyMessageStore(db_url=settings.database_url)
await message_store.init_db()

run_store = InMemoryRunStore()
checkpoint_store = InMemoryCheckpointStore()
event_sink = InMemoryEventSink()
runner = PipelineRunner(
    event_sink=event_sink,
    run_store=run_store,
    checkpoint_store=checkpoint_store,
    execution_policy=ExecutionPolicy(timeout_seconds=settings.default_timeout_seconds),
)
pipeline = SupportMiniAppPipeline(
    driver=driver,
    ask_model=ask_model,
    message_store=message_store,
)

{
    'database_url': settings.database_url,
    'driver': type(driver).__name__,
    'model': settings.default_model,
    'timeout_seconds': settings.default_timeout_seconds,
}

## 4. Run único

Primeiro, um fluxo de cobrança para validar a rota `billing`.


In [ ]:
single_context = RunContext(
    run_id='support-run-001',
    correlation_id='corr-support-001',
    started_at=datetime.now(timezone.utc),
    input_payload={
        'prompt': 'Minha fatura veio duplicada e preciso de ajuda.',
        'customer_tier': 'enterprise',
    },
    metadata={'channel': 'notebook-demo'},
)

single_result = await runner.run_pipeline(pipeline, single_context)
single_result


## 5. Auditoria do run

O fluxo novo permite inspecionar resultado, run status, checkpoint e eventos emitidos.


In [ ]:
{
    'run_record': await run_store.get_run('support-run-001'),
    'checkpoint': await checkpoint_store.get_checkpoint('support-run-001'),
    'events': [event.type for event in event_sink.events if event.run_id == 'support-run-001'],
}


## 6. Batch de runs

Agora repetimos o fluxo com cenários de suporte e geral, sempre usando o Gemini CLI real.


In [ ]:
scenarios = [
    ('support-run-002', 'Recebi um erro ao acessar o painel.', 'standard'),
    ('support-run-003', 'Preciso confirmar um pagamento pendente.', 'standard'),
    ('support-run-004', 'Quero apenas entender o status do meu pedido.', 'standard'),
]

batch_results = []
for run_id, prompt, tier in scenarios:
    context = RunContext(
        run_id=run_id,
        correlation_id=f'corr-{run_id}',
        started_at=datetime.now(timezone.utc),
        input_payload={'prompt': prompt, 'customer_tier': tier},
        metadata={'channel': 'batch-demo'},
    )
    result = await runner.run_pipeline(pipeline, context)
    run_record = await run_store.get_run(run_id)
    checkpoint = await checkpoint_store.get_checkpoint(run_id)
    batch_results.append({
        'run_id': run_id,
        'route': result['route'],
        'priority': result['priority'],
        'status': run_record['status'],
        'reply_excerpt': checkpoint['reply'][:120],
    })

batch_results


## 7. Mensagens persistidas

As mensagens ficam em store separado do lifecycle do runtime. Isso continua verdadeiro mesmo com o Gemini CLI como motor.


In [ ]:
messages = await message_store.get_messages()
[
    {
        'id': message.id,
        'sender_id': message.sender_id,
        'content': message.content[:120],
        'run_id': message.additional_info.get('run_id'),
        'route': message.additional_info.get('route'),
    }
    for message in messages
]


## 8. Eventos do runtime

O `PipelineRunner` continua emitindo eventos canônicos. O motor mudou, mas a observabilidade do runtime segue intacta.


In [ ]:
[
    {
        'type': event.type,
        'run_id': event.run_id,
        'scope': event.scope,
        'payload': event.payload,
    }
    for event in event_sink.events
]


## Extensão sugerida

Próximo passo natural: trocar o store de runs/checkpoints em memória por persistência real, mantendo o mesmo `AgentAPIDriver` com Gemini CLI.

In [ ]:
await client.close()
'driver client closed'